# 3_1 Gold — Leads Activities Summary

Builds two Gold tables per spec Section 3:

1. **STAFF_DIM** — Deduplicated staff/user dimension with real names/emails
   - Coaches, CSMs, setters/closers, and partner accounts (everyone who can act in Close CRM)
   - Replaces masked user hashes (`user_<43hex>`) with real human names
   - Used by LEADS_ACTIVITIES_SUMMARY for name enrichment and by SALES_DETAILS for setter/closer
   - Dedup: `USER_ID`, latest by `DATE_UPDATED DESC`

2. **LEADS_ACTIVITY_FACT** — Deduplicated activity fact table enriched with real staff names
   - One row per ACTIVITY_ID (latest by `DATE_UPDATED DESC → INSERT_DATE DESC`) — collapses silver's CDC version history into current state
   - Attributed to staff via staff_dim (`created_by` hash → real name/email)

In [0]:
%sql
-- # STAFF_DIM — Deduplicated staff/user dimension with real names/emails
-- Coaches, CSMs, setters/closers, and partner accounts (everyone who can act in Close CRM)
-- Replaces masked user hashes (user_<43hex>) with real human names
-- Used by LEADS_ACTIVITIES_SUMMARY for name enrichment and by SALES_DETAILS for setter/closer
-- Dedup: USER_ID, latest by DATE_UPDATED DESC
CREATE OR REPLACE TABLE crm_ingestion.gold.staff_dim AS (
    SELECT 
        csm_id,
        image,
        email,
        first_name,
        last_name,
        date_created,
        date_updated,
        bronze_insert_date,
        full_name AS csm_name,
        email_normalized AS csm_email
    FROM (
        SELECT
            id AS csm_id,
            image,
            email,
            first_name,
            last_name,
            date_created,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn,
            CONCAT_WS(' ',first_name, last_name) AS full_name,
            LOWER(TRIM(email)) AS email_normalized
        FROM
            crm_ingestion.silver.close_crm_users_processed      
)
    WHERE
        csm_id IS NOT NULL
    AND 
        rn = 1
)


In [0]:
%sql
SELECT 
    *
FROM
    crm_ingestion.gold.staff_dim
LIMIT 10

In [0]:
%sql
DESCRIBE TABLE crm_ingestion.silver.lead_activites_processed